# Pipeline Generalizable de ML para Ames Housing

Este notebook construye un pipeline robusto y reutilizable con sklearn y feature-engine, implementando todas las transformaciones identificadas en el análisis previo (Feature Engineering, notebook 02). El pipeline es 100% tabular: detecta y actúa sobre grupos de variables dinámicamente.

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, AddMissingIndicator
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer, SklearnTransformerWrapper
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
import joblib

---

## 2. Carga del dataset y Train/Test Split

In [ ]:
# Configuración
DATA_PATH = 'data/train.csv'
TARGET = 'SalePrice'

# Cargar y eliminar Id
X = pd.read_csv(DATA_PATH)
if 'Id' in X.columns:
    X = X.drop('Id', axis=1)

y = X[TARGET].copy()
X = X.drop(TARGET, axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

---

## 3. Transforma el target con np.log (regresión)

In [ ]:
# SalePrice es continua, log-transform
y_train = np.log(y_train)
y_test = np.log(y_test)

---

## 4. Declaración de CONSTANTES

In [ ]:
# Listas obtenidas directamente del feature engineering anterior
CATEGORICAL_VARS_WITH_NA_MISSING = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtFinType2',
    'BsmtExposure', 'BsmtFinType1', 'BsmtCond', 'BsmtQual'
]
CATEGORICAL_VARS_WITH_NA_FREQUENT = ['Electrical']
NUMERICAL_VARS_WITH_NA = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
TEMPORAL_VARS = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
REF_VAR = 'YrSold'
NUMERICALS_LOG_VARS = [
    'MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF',
    'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath',
    'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage',
    'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF',
    'GrLivArea', 'BsmtUnfSF', '2ndFlrSF'   # quitamos SalePrice
]
BINARIZE_VARS = [
    'MasVnrArea', 'BsmtFinSF2', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath',
    'BsmtHalfBath', 'HalfBath', 'WoodDeckSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal'
]
# Quality mappings ordinales
QUAL_VARS = [
    'ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
    'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC'
]
QUAL_MAPPINGS = {
    'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0
}
# Todas categóricas nominativas (para rare/ordinal encoder)
CATEGORICAL_VARS = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
    'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
    'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition'
]

---

## 5. Custom Transformers

In [ ]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    """
    Calcula años transcurridos: feature = reference_variable - feature
    """
    def __init__(self, variables, reference_variable):
        self.variables = variables
        self.reference_variable = reference_variable

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var + '_elapsed'] = X[self.reference_variable] - X[var]
        return X

class Mapper(BaseEstimator, TransformerMixin):
    """
    Para mapear variables ordinales con diccionarios
    """
    def __init__(self, variables, mappings):
        self.variables = variables
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var + '_num'] = X[var].map(self.mappings).astype(float)
        return X

---

## 6. Construcción del Pipeline de sklearn + feature-engine

In [ ]:
# Identificamos las variables de mapeo nominal vs ordinal
def get_nominal_vars():
    # CATEGORICAL_VARS menos QUAL_VARS
    return [c for c in CATEGORICAL_VARS if c not in QUAL_VARS]

# Pipeline
pipeline = Pipeline([
    # 1-IMPUTATION
    ('categorical_missing_imputer', 
        CategoricalImputer(imputation_method='missing', variables=CATEGORICAL_VARS_WITH_NA_MISSING)),
    ('categorical_frequent_imputer', 
        CategoricalImputer(imputation_method='frequent', variables=CATEGORICAL_VARS_WITH_NA_FREQUENT)),
    ('add_missing_indicator', 
        AddMissingIndicator(variables=NUMERICAL_VARS_WITH_NA)),
    ('mean_imputer', 
        MeanMedianImputer(imputation_method='mean', variables=NUMERICAL_VARS_WITH_NA)),
    # 2-TEMPORAL
    ('temporal_elapsed', 
        TemporalVariableTransformer(variables=TEMPORAL_VARS, reference_variable=REF_VAR)),
    ('drop_ref_var', 
        # Simple custom dropper inline
        FunctionTransformer(lambda X: X.drop(columns=[REF_VAR]), validate=False)
    ),
    # 3-TRANSFORMATION
    ('log_transformer', LogTransformer(variables=NUMERICALS_LOG_VARS)),
    ('binarizer', SklearnTransformerWrapper(
        Binarizer(threshold=0), variables=BINARIZE_VARS)),
    # 4-MAPPERS
    ('quality_mapper', Mapper(variables=QUAL_VARS, mappings=QUAL_MAPPINGS)),
    # 5-ENCODING
    ('rare_encoder', RareLabelEncoder(variables=get_nominal_vars(), tol=0.01, n_categories=1)),
    ('ordinal_encoder', OrdinalEncoder(variables=get_nominal_vars(), encoding_method='ordered', ignore_format=True)),
    # 6-SCALING
    ('scaler', SklearnTransformerWrapper(MinMaxScaler(), variables=None))
])

---

## 7. Ajuste pipeline SOLO en train

In [ ]:
pipeline.fit(X_train, y_train)

---

## 8. Transformación de train y test

In [ ]:
X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

---

## 9. Chequeo: no hay NaN / infinito

In [ ]:
if isinstance(X_train_transformed, np.ndarray):
    X_train_check = pd.DataFrame(X_train_transformed)
    X_test_check = pd.DataFrame(X_test_transformed)
else:
    X_train_check = X_train_transformed.copy()
    X_test_check = X_test_transformed.copy()
assert not np.any(pd.isnull(X_train_check)), 'Train contiene NaN!'
assert not np.any(pd.isnull(X_test_check)), 'Test contiene NaN!'
assert np.isfinite(X_train_check).all().all(), 'Train contiene infinito!'
assert np.isfinite(X_test_check).all().all(), 'Test contiene infinito!'
print('✔️ Transformación completa. Sin NaN ni infinito en train/test.')

---

## 10. Serialización pipeline (joblib)

In [ ]:
joblib.dump(pipeline, 'proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')
print('✔️ Pipeline serializado en proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')